# 00 · Warm-up: what's wrong with this notebook?

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 30 minutes live | [P0 · Python refresher (pre-work)](P0_python_refresher.ipynb) | [01 · Classes and objects](01_classes_and_objects.ipynb) |

Below is a churn-prediction notebook of the kind that gets written under time pressure, when the priority is "make it work" rather than "make it last". It loads real customer data from a telecom company, trains a model and prints a report.

And it works. Run it top to bottom and you get sensible numbers.

But it hides problems, one of them serious enough to send the wrong customers to the retention team. Over the next two days you will learn the tools to fix every one of them. Today, you only need to find them.

> **Learning goals.** By the end you can:
>
> - point out concrete places where notebook code becomes hard to reuse, test or run in production
> - describe a bug that produces wrong results without any error message
> - explain your findings to a partner in plain words

**How we work (Think-Pair-Share).** 5 minutes alone: read and run the notebook. 10 minutes with your buddy: collect problems. Then we share in the group.

---

## The notebook

The business question: which customers are likely to cancel their contract next month, so the retention team can contact them with an offer?

In [ ]:
# --- Cell 1: load data ---
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/raw/telco_churn.csv")
print(df.shape)
df.head()

In [ ]:
# --- Cell 2: quick look ---
df.info()

In [ ]:
# --- Cell 3: TotalCharges is text for some reason, fix it ---
df2 = df.copy()
df2["TotalCharges"] = pd.to_numeric(df2["TotalCharges"], errors="coerce")
df2 = df2.dropna()
print(df2.shape)

In [ ]:
# --- Cell 4: encode ---
le = LabelEncoder()
df2["Contract"] = le.fit_transform(df2["Contract"])
df2["InternetService"] = le.fit_transform(df2["InternetService"])
df2["PaymentMethod"] = le.fit_transform(df2["PaymentMethod"])
df2["PaperlessBilling"] = le.fit_transform(df2["PaperlessBilling"])
df2["Churn"] = le.fit_transform(df2["Churn"])
# there are more columns but let's just use these for now

In [ ]:
# --- Cell 5: features and target ---
features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen",
    "Contract",
    "InternetService",
    "PaymentMethod",
    "PaperlessBilling",
]
X = df2[features]
y = df2["Churn"]

In [ ]:
# --- Cell 6: split and train ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=42)
model.fit(X_train, y_train)

In [ ]:
# --- Cell 7: evaluate ---
preds = model.predict(X_test)
print(classification_report(y_test, preds))

A recall of about 0.5 on the churners, an accuracy near 0.8. Not bad for a first model. The notebook goes into the team's shared folder.

## One month later

The CRM system exports the customers whose **contract ends next month**. Only one-year and two-year contracts have an end date, so the export contains only those. The retention team wants to know whom to call. A colleague adds two cells:

In [ ]:
# --- Cell 8: score next month's renewals ---
raw = pd.read_csv("../data/raw/telco_churn.csv")
renewals = raw[raw["Contract"] != "Month-to-month"].sample(
    200, random_state=7
)  # stands in for the CRM export

new = renewals.copy()
new["TotalCharges"] = pd.to_numeric(new["TotalCharges"], errors="coerce")
new = new.dropna()
new["Contract"] = le.fit_transform(new["Contract"])
new["InternetService"] = le.fit_transform(new["InternetService"])
new["PaymentMethod"] = le.fit_transform(new["PaymentMethod"])
new["PaperlessBilling"] = le.fit_transform(new["PaperlessBilling"])

risk = model.predict_proba(new[features])[:, 1]
print(f"Customers to call (risk above 30 %): {(risk >= 0.3).sum()}")
print(f"Average churn risk in this batch: {risk.mean():.1%}")

The team calls those customers. Nothing crashed, no warning was printed.

## Something does not add up

These 200 customers are also in our historical table, which we prepared in cells 3 and 4. Let's score the very same rows from there:

In [ ]:
# --- Cell 9: sanity check, same customers, same model ---
same_rows = df2.loc[new.index, features]
risk_check = model.predict_proba(same_rows)[:, 1]
print(f"Customers to call (risk above 30 %): {(risk_check >= 0.3).sum()}")
print(f"Customers whose call decision differs: {((risk >= 0.3) != (risk_check >= 0.3)).sum()}")
print(f"Average churn risk: {risk_check.mean():.1%}")
print(f"Largest difference for a single customer: {np.abs(risk - risk_check).max():.2f}")

Same model, same customers, different answers. The average risk roughly doubles, and the list of customers to call is about four times longer. Most of the extra calls go to customers the model, scored correctly, considers safe.

And here is the part that makes it hard to catch: try it with a **random** sample of customers instead of the renewals.

In [ ]:
# --- Cell 10: same procedure, random customers ---
batch = raw.sample(200, random_state=7)
b = batch.copy()
b["TotalCharges"] = pd.to_numeric(b["TotalCharges"], errors="coerce")
b = b.dropna()
for col in ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling"]:
    b[col] = le.fit_transform(b[col])

diff = np.abs(
    model.predict_proba(b[features])[:, 1] - model.predict_proba(df2.loc[b.index, features])[:, 1]
)
print(f"Largest difference on a random batch: {diff.max():.2f}")

On a random batch everything matches. A quick test by the colleague would have passed.

**The mystery: why do the renewals get different scores, and why only them?** Don't look for the answer online yet. Discuss it with your buddy; we solve it together in notebook 04.

---
## Your task (Think-Pair-Share)

With your buddy, collect every place where this notebook would hurt you or your team. Write them down in the cell below. Some questions to guide you:

- **Reuse**: what would you have to copy-paste to train on a new data extract, or to try a second model?
- **Change**: what breaks if the CRM renames a column, or adds a new payment method?
- **Readability**: what do `df2`, `le` and the numbers `200`, `10`, `0.3` mean, and who decides them?
- **Correctness**: what happened to the 11 rows dropped in cell 3? Who are those customers? (Hint: look at their `tenure`.)
- **Testing**: how would you prove that cell 8 is correct before the retention team acts on it?
- **Production**: could this run unattended every month? What would happen if the file were missing?

*Your notes:*

-
-
-

<details><summary><b>Some of the problems trainers usually hear (open after your discussion)</b></summary>

- Preprocessing is copy-pasted between training (cells 3 and 4) and scoring (cell 8). Any change must be made twice, and it already diverged.
- One `LabelEncoder` object is re-fitted again and again, each time on different data. What it "knows" at the end depends on the last column it saw.
- Magic numbers (`0.2`, `42`, `200`, `10`, `0.3`) and a hard-coded path, with no explanation of why.
- `dropna()` silently removes 11 customers. They have `tenure == 0`: they joined this month and have not been billed yet. Their `TotalCharges` should be 0, not "missing", and dropping them means the model never sees brand-new customers.
- Variables like `df`, `df2`, `X`, `new`, `b` are overwritten as the notebook runs. Running the cells in a different order gives different results.
- There is nothing to test: no functions, no classes, no single place that defines "how a customer is prepared for the model".
- Nothing tells you when something goes wrong: no validation of the input columns, no logging, and the scoring bug produced no error at all.
- The trained model and its preprocessing live only in memory. Next month someone has to re-run the whole notebook.

</details>

---

## Where we fix each problem

| Problem | Tool | Notebook |
|---|---|---|
| Loose variables, nothing bundled | classes and objects | 01 |
| Copy-pasted logic for similar models | inheritance | 02 |
| Results that are hard to inspect and combine | special methods | 03 |
| Preprocessing that forgets what it learned (the mystery) | encapsulation, abstract base classes | 04 |
| Code that only works with one kind of model | polymorphism, protocols | 05 |
| Magic numbers, unclear inputs | type hints, docstrings, dataclasses | 07 |
| One big blob doing everything | design and architecture | 08 |
| Silent failures | exceptions and logging | 09 |
| "It worked when I tried it" | testing | 10 |
| Code that only lives in a notebook | packages | 11 |
| All of the above | capstone | 12 |

## Summary

- Code that runs and produces plausible numbers can still be wrong. The worst bugs in data science raise no error.
- Most problems here are not about the model. They are about how the code is organised: what is bundled together, what is repeated, and what remembers what.
- Keep your list. In the capstone (notebook 12) you will turn this notebook into a small, tested package, and the acceptance tests check exactly the bug you just found.

---

## Sources and further reading

**Sources.** This notebook is based on:

- [scikit-learn: Common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html), in particular "Inconsistent preprocessing". This is the bug hidden in this notebook.
- [Sculley et al. (2015), *Hidden Technical Debt in Machine Learning Systems*](https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html), NeurIPS. The classic paper on why ML code rots: glue code, pipeline jungles, configuration debt.

**For self-study.** Where to go next if you want to learn more:

- [Wilson et al. (2017), *Good enough practices in scientific computing*](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005510), PLOS Computational Biology (paper, beginner): a short, practical checklist written for researchers. Ideal if you come from academia.
- [Joel Grus, *I don't like notebooks*](https://www.youtube.com/watch?v=7jiPeIFXb6U) (talk, JupyterCon 2018, 50 min): an entertaining look at the hidden-state problems of notebooks.
- [Google: Rules of Machine Learning](https://developers.google.com/machine-learning/guides/rules-of-ml) (guide, intermediate): engineering lessons from production ML, including training/serving skew.

---

[← P0 · Python refresher (pre-work)](P0_python_refresher.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [01 · Classes and objects →](01_classes_and_objects.ipynb)